### Replace 없는 최종 버전

In [ ]:
import numpy as np

# -----------------------------
# 0) Tokenizer (byte-level)
# -----------------------------
BOS = 256           # begin of sequence
EOS = 257           # end of sequence
VOCAB_SIZE = 258  # 0~255(토큰 범위) bytes + BOS/EOS: 총 258개의 토큰

def encode(text: str):  # 문자열 -> 정수 리스트
    bs = text.encode("utf-8", errors="replace") # 문자열 -> UTF-8바이트 배열 변환(전 세계 문자 처리 가능), UTF-8로 변환 못하는 문자가 있으면 �로 대체
                                                # UTF-8: 유니코드 문자를 바이트로 표현하는 규칙
    return [BOS] + list(bs)    # 문장 앞에 시작 토큰을 붙임, 바이트열을 정수 리스트로 변환

def decode_ascii(tokens):
  bs = []
  for t in tokens:
    if t in (BOS, EOS):
      continue
    if 32 <= t <= 126:
      bs.append(t)
  return bytes(bs).decode("ascii", errors = "ignore")

# -----------------------------
# 1) Config
# -----------------------------
class GPTConfig:  # 트랜스포머의 크기와 구조를 정의하는 설정값 객체
    def __init__(self, vocab_size=VOCAB_SIZE, n_layers=2, d_model=128, n_heads=4, d_ff=512, max_seq=256, ln_eps=1e-5):
        assert d_model % n_heads == 0  # head마다 동일한 차원을 나눠 가지게 하기 위함, 잘못된 설정 조기 차단
        self.vocab_size = vocab_size  # 토큰 종류 개수
        self.n_layers = n_layers  # decorder block 개수 = 깊이
        self.d_model = d_model  # 토큰 임베딩 벡터 차원, 트랜스포머의 가로 폭
        self.n_heads = n_heads  # MHA의 head 수
        self.d_head = d_model // n_heads  # head 하나당 차원
        self.d_ff = d_ff   # FFN 내부 히든 차원
        self.max_seq = max_seq   # attention / KV cache 크기 상한
        self.ln_eps = ln_eps   # LayerNorm 안정화용 작은 수 입실론, 분산이 0에 가까울 때 division-by-zero 방지

# -----------------------------
# 2) Core ops
# -----------------------------
def layernorm(x, g, b, eps=1e-5):
    # x: (D,)
    mu = x.mean()   # 모든 원소 평균, 반환값은 스칼라
    var = ((x - mu) ** 2).mean()  # 분산
    xhat = (x - mu) / np.sqrt(var + eps) # 표준편차로 나누어 정규화
    return xhat * g + b # 스케일과 bias 적용

def gelu(x):
    # GPT-2 style approximation
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0/np.pi) * (x + 0.044715 * (x**3))))  #GELU 근사식

def softmax(logits):  #logits: (V, )
    m = np.max(logits)  # logits 중 최댓값 -> 수치 안정성을 위해 필요
    exps = np.exp(logits - m)  # overflow 방지, softmax 결과는 동일
    s = np.sum(exps)  # 모든 exps 값의 합
    return exps / (s + 1e-12)  # 확률로 정규화, 1e-12: 분모가 0이되는 극단적 상황 방지

def top_k_filter(logits, k):  # top-K: 상위 k개 후보만 남기고 나머지 제거, 샘플링 시 노이즈 증가 방지 -> 생성 품질 안정화
    if k <= 0 or k >= logits.shape[0]:  # k가 0이거나 vocab 이상이면 필터링X, shape[0] = V: 첫번째 차원의 크기
        return logits
    out = logits.copy()  # 원본 logits 보존, 새로운 배열에서 수정
    idx = np.argpartition(out, -k)[:-k]  # top-k를 제외한 나머지 인덱스(지울 대상), 큰 값 기준으로 k개를 뒤쪽에 모음, [:-k]: 뒤에서 k개를 제외한 나머지 전부
    out[idx] = -1e30  # top-k 밖의 logits을 매우 작은 값으로 설정 사실상 0
    return out

def sample_from_probs(probs):  #sampling: 확률에 비례해 무작위 선택, 각 토큰이 자기 확률만큼의 구간을 가지도록 만듦
    # probs: (V,) : 토큰이 선택될 확률, 길이가 V인 1차원 배열
    r = np.random.rand()  # [0, 1) 범위의 균등 난수
    c = np.cumsum(probs)  # 누적 확률 분포(CDF), 누적합 c[i] = 토큰 i까지 뽑힐 확률의 총합
    return int(np.searchsorted(c, r, side="right"))  # r이 들어갈 위치를 c(CDF)에서 찾음, 해당 index = 선택된 토큰, side="right": 같은 값의 오른쪽(경계값 표현을 위해)

# -----------------------------
# 3) RoPE (Rotary Positional Embedding)
# -----------------------------
def rope_apply(q, k, pos):  # q, k 벡터의 차원을 2개씩 묶어서 위치 pos에 따라 2D 회전 적용한 위치정보를 벡터 자체에 섞어넣는 함수
    """
    q,k: (Dh,)  Dh must be even
    pos: int
    """
    Dh = q.shape[0]  # Dh: head dimension, RoPE을 적용할 벡터의 실제 차원 수
    assert Dh % 2 == 0  # 2개씩 묶어 회전해야하므로 Dh는 항상 짝수여야함
    q2 = q.copy()  # .copy는 깊은 복사로, q, k를 직접 바꾸지 않고 새 배열에 적용
    k2 = k.copy()

    # rotate pairs (2i, 2i+1)
    for i in range(0, Dh, 2): # (i, i+1)을 한 쌍으로 처리하기 위해 2씩 점프
        idx = i // 2  # i가 몇 번째 pair 인지를 나타냄
        theta = pos / (10000.0 ** (2.0 * idx / Dh))  # pos를 곱하는 이유: 토큰이 뒤에 있을수록 더 많이 회전시켜라
        c = np.cos(theta)
        s = np.sin(theta)

        a, b = q2[i], q2[i+1]
        q2[i]   = a * c - b * s
        q2[i+1] = a * s + b * c

        a, b = k2[i], k2[i+1]
        k2[i]   = a * c - b * s
        k2[i+1] = a * s + b * c

    return q2, k2

# -----------------------------
# 4) Weights (random init + optional load/save)
# -----------------------------
def init_weights(cfg: GPTConfig, seed=42):  # cfg에 정의된 크기(d_model, n_layers, d_ff, vocab_size)에 맞춰 Token Embedding,
                                            # 각 레이어의 LN/Attention/FFN 가중치, 최종 LN, 출력 헤드를 랜덤 초기화해서 dict로 반환한다.
                                            # seed = 42: 랜덤 초기화가 항상 동일하게 나오도록(재현성)
    rng = np.random.default_rng(seed)  # rng: NumPy의 새 난수 생성기
    D = cfg.d_model  # 토큰 표현 벡터 길이
    V = cfg.vocab_size  # 토큰 개수

    W = {}  # 가중치 저장 딕셔너리, 모든 파라미터를 이 딕셔너리에 저장하여 접근
    # token embedding: (V, D)
    W["tok_emb"] = rng.normal(0, 0.02, size=(V, D)).astype(np.float32)  # 난수로 (V, D)배열 생성, 평균 0, 표준편차 0.02인 정규 분포에서 샘플링
                                                                        # 크기 V행 D열, dtype을 float32로 변화, 딕셔너리에 저장, x = W["tok_emb"][token_id]가 가능하도록 lookup table 만듦
    W["blocks"] = []  # 레이어별 블록 생성, blocks: 레이어 개수만큼의 리스트
    for _ in range(cfg.n_layers):
        blk = {}
        # LN params
        blk["ln1_g"] = np.ones((D,), dtype=np.float32)  # 한 블록에서 보통 LN이 2번(Attention 앞, FFN 앞) 들어감(pre-LN)
        blk["ln1_b"] = np.zeros((D,), dtype=np.float32) # g(gamma): 스케일 파라미터, 보통 1로 시작
        blk["ln2_g"] = np.ones((D,), dtype=np.float32)  # b(beta): 시프트 파라미터, 보통 0으로 시작
        blk["ln2_b"] = np.zeros((D,), dtype=np.float32) # np.ones((D,)) : 길이 D짜리 벡터를 전부 1로, np.zeros((D,)) : 길이 D짜리 벡터를 전부 0으로

        # Attention projections: use (D, D) so x @ W gives (D,)
        scale = 0.02
        blk["Wq"] = rng.normal(0, scale, size=(D, D)).astype(np.float32)  # rng.normal(mean, std, size=...): 정규분포(가우시안)난수를 뽑아주는 함수
        blk["Wk"] = rng.normal(0, scale, size=(D, D)).astype(np.float32)  # mean=0: 평균 0, std=0.02: 값 너무 안 커지게하는 분산, size: 배열크기
        blk["Wv"] = rng.normal(0, scale, size=(D, D)).astype(np.float32)  # 작은 랜덤 값으로 서로 다른 역할을 갖게 만듦
        blk["Wo"] = rng.normal(0, scale, size=(D, D)).astype(np.float32)

        blk["bq"] = np.zeros((D,), dtype=np.float32)
        blk["bk"] = np.zeros((D,), dtype=np.float32)
        blk["bv"] = np.zeros((D,), dtype=np.float32)
        blk["bo"] = np.zeros((D,), dtype=np.float32)

        # FFN = W2*GELU(W1x + b1) + b2
        blk["W1"] = rng.normal(0, scale, size=(D, cfg.d_ff)).astype(np.float32)
        blk["b1"] = np.zeros((cfg.d_ff,), dtype=np.float32)
        blk["W2"] = rng.normal(0, scale, size=(cfg.d_ff, D)).astype(np.float32)
        blk["b2"] = np.zeros((D,), dtype=np.float32)

        W["blocks"].append(blk)  # 블록을 리스트에 추가, 레이어별 파라미터를 리스트에 차례대로 저장

    # final LN: 모든 블록을 지난 후 마지막에 한 번 더 정규화 -> 출력 분포 안정화, pre-LN의 출력을 정돈
    W["ln_f_g"] = np.ones((D,), dtype=np.float32)
    W["ln_f_b"] = np.zeros((D,), dtype=np.float32)

    # output head: (D, V) hidden state(D, ) -> 점수(logits)
    W["head_W"] = rng.normal(0, 0.02, size=(D, V)).astype(np.float32)
    W["head_b"] = np.zeros((V,), dtype=np.float32)

    return W  # 모든 파라미터를 dict로 변환하여 inference 코드가 사용하게 됨

def save_weights_npz(path, W):  # Python dict에 들어 있는 NumPy 배열들을 하나의 파일 .npz(NumPy의 압축된 다중 배열 포맷)로 저장
                                # save 함수는 중첩된 weights dict를 b{layer}_{param} 규칙의 flat key로 변환해 .npz 파일에 저장한다.
    # Flatten dict for np.savez
    to_save = {
        "tok_emb": W["tok_emb"],
        "ln_f_g": W["ln_f_g"],
        "ln_f_b": W["ln_f_b"],
        "head_W": W["head_W"],
        "head_b": W["head_b"],
    }
    for i, blk in enumerate(W["blocks"]): # i: layer index, blk: i번째 레이어의 dict, enumerate(W["blocks"]): 리스트의 (index, value)쌍을 차례대로 반환
        for k, v in blk.items():
            to_save[f"b{i}_{k}"] = v  # 레이어 번호 + 파라미터 이름을 합쳐서 고유한 key 생성
    np.savez(path, **to_save)  # **to_save: dict를 keyword argument로 펼침 -> path.npz 파일 안에 {key: array} 쌍들이 저장됨
                               # np.savez(): 여러 개의 NumPy 배열을 하나의 .npz 파일로 저장하는 함수

def load_weights_npz(path, cfg: GPTConfig):  # .npz 파일에서 배열들을 읽어 원래 weights dict 구조로 정확히 복원함, path: 저장된 .npz 파일, cfg: 모델 구조 정보(레이어 개수 등)
    z = np.load(path)  # .npz 파일을 열면 z는 dict처럼 동작하는 객체
    W = {"blocks": []}  # 새로운 weight dict 생성, 블록 리스트부터 준비
    W["tok_emb"] = z["tok_emb"]  # save할 때 와 동일한 key 이름으로 불러옴
    W["ln_f_g"] = z["ln_f_g"]
    W["ln_f_b"] = z["ln_f_b"]
    W["head_W"] = z["head_W"]
    W["head_b"] = z["head_b"]
    for i in range(cfg.n_layers):  # config에 정의된 레이어 수만큼 복원
        blk = {}  # 한 레이어(block)의 모든 파라미터(가중치)를 담을 그릇
        for k in ["ln1_g","ln1_b","Wq","Wk","Wv","Wo","bq","bk","bv","bo","ln2_g","ln2_b","W1","b1","W2","b2"]:  # 한 Transformer block이 반드시 가져야 할 파라미터의 정의 목록을 어떤 순서/이름으로 로드할지 고정한 것
            blk[k] = z[f"b{i}_{k}"]  # save 단계에서 만든 key 규칙대로 배열을 다시 꺼냄
        W["blocks"].append(blk)  # 완성된 레이어 파라미터를 리스트에 추가
    return W  # 원래와 같은 구조의 weight dict 반환

# -----------------------------
# 5) KV Cache
# -----------------------------
class KVCache:
    """
    For each layer:
      K: (T, H, Dh)
      V: (T, H, Dh)
    We append one step at a time.
    """
    def __init__(self, cfg: GPTConfig):  # 모델 설정(cfg)을 받아서 레이어 수, head 수 등을 맞춤, self는 만들어지고 있는 객체(cache) 자신
        self.cfg = cfg
        self.K = [None for _ in range(cfg.n_layers)]  # self.K, self.V는 리스트, 길이는 레이어 개수
        self.V = [None for _ in range(cfg.n_layers)]  # 처음엔 [none, none, none], 이후 리스트 안의 원소가 (T, H, Dh) NumPy 배열로 바뀜
        self.T = 0  # current cached length(현재까지 캐시된 토큰 수, 시퀀스 길이 T)

    def append(self, layer, k_h, v_h):  # 한 토큰이 새로 생성될 때마다 호출됨, k_h, v_h: 현재 토큰의 K, V, 모든 head에 대한 값
        # k_h, v_h: (H, Dh) for current position
        assert 0 <= layer < self.cfg.n_layers  # layer 범위 체크
        assert self.T < self.cfg.max_seq  # 캐시 길이가 max_seq를 넘지 않도록 체크
        # k_h/v_h shape 체크 (H, Dh)
        assert k_h.shape == (self.cfg.n_heads, self.cfg.d_head)
        assert v_h.shape == (self.cfg.n_heads, self.cfg.d_head)

        if self.K[layer] is None:
            self.K[layer] = k_h[None, :, :]  # (1,H,Dh) 앞에 차원 추가, 1이 시간축(T): 토큰 1개짜리
            self.V[layer] = v_h[None, :, :]
        else:  # 이미 캐시가 있는 경우(T >= 1)
            self.K[layer] = np.concatenate([self.K[layer], k_h[None, :, :]], axis=0)  #(T, H, Dh) + (1, H, Dh) = (T+1, H, Dh)
            self.V[layer] = np.concatenate([self.V[layer], v_h[None, :, :]], axis=0)  # 실무에서는 최대 길이 max_seq로 (max_seq, H, Dh)로 할당해두고 self.T 위치에 바로 대입하는 방식(append 없이 in-place)을 많이 쓴다

    def step_done(self):  # 한 토큰 생성이 완전히 끝났음을 표시
        self.T += 1  # 캐시 길이(T) 증가, T는 현재 몇 토큰까지 왔는지 추적하는 메타 정보

# -----------------------------
# 6) One-step forward (decoder-only)
# -----------------------------
# x_In으로부터 Q/k/V를 만들고 RoPE 적용, KV cache 업데이트하고 과거 토큰들에 대한 attention 가중합 결과 out을 반환
def attention_step(cfg: GPTConfig, blk, x, pos, cache: KVCache, layer: int):  # pos: 현재 토큰의 위치 인덱스(RoPE용) blk: 트랜스포머 한 블록의 파라미터(dict)
    """
    x: (D,)
    returns: (D,)  # attention 결과 벡터(잔차에 더해질 값)
    """
    D, H, Dh = cfg.d_model, cfg.n_heads, cfg.d_head

    assert 0 <= pos < cfg.max_seq  # pos 범위 체크
    assert cache.T == pos  # incremental decoding에서 pos와 cache.T가 일치해야 함

    # Projections: (D,) @ (D,D) -> (D,)
    q = x @ blk["Wq"] + blk["bq"]  # 현재 토큰 1개에 대한 hidden vector x를 Q, K, V로 각각 변환: x라는 한 벡터를 attention에서
    k = x @ blk["Wk"] + blk["bk"]  #  Q, K, V로 쓰기 좋은 표현 공간으로 다시 투영한 것
    v = x @ blk["Wv"] + blk["bv"]

    # reshape to heads ex) D = 768, H = 12, Dh = 64 -> q.reshape(H, Dh) = (12, 64)
    q = q.reshape(H, Dh)  # 각 head가 독립적으로 attention을 수행하도록 Multi-Head 형태로 reshape: 독립적으로 계산되도록 head 차원을 분리
    k = k.reshape(H, Dh)  # Wq가 만들어낸 (D, ) 결과를 head 단위로 나눠쓰는 구조
    v = v.reshape(H, Dh)

    # RoPE on each head for q,k
    for h in range(H):
        q[h], k[h] = rope_apply(q[h], k[h], pos)

    # append k,v to cache for this layer
    cache.append(layer, k.astype(np.float32), v.astype(np.float32))  # K와 V를 float32로 고정 -> 안정성 상승, 메모리 절약

    # Now attend over 0..T (inclusive current), which is cache length after append:
    K = cache.K[layer]  # (T+1, H, Dh)
    V = cache.V[layer]  # (T+1, H, Dh)
    T = K.shape[0]  # 첫번째 축 길이 = 현재 레이어 cache에 저장된 토큰 개수 = 시퀀스 길이

    assert K.shape[0] <= cfg.max_seq  # cache 내부 배열이 max_seq를 넘지 않는지 체크

    # scores[t] = q·K[t] / sqrt(Dh), per head, scaled dot product attention
    scale = 1.0 / np.sqrt(Dh)

    out = np.zeros((H, Dh), dtype=np.float32)
    for h in range(H):
        # (T, Dh) dot (Dh,) => (T,)
        scores = (K[:, h, :] @ q[h]) * scale  # attention score
        probs = softmax(scores)  # 각 과거 토큰이 얼마나 중요한지
        # weighted sum: (T,) @ (T,Dh) => (Dh,)
        out[h] = probs @ V[:, h, :]  # attention output

    out = out.reshape(D)  # merge heads, muti-head attention의 결과를 하나의 벡터로 합침:(H, Dh) -> (D, )
    out = out @ blk["Wo"] + blk["bo"]  # Wo: head 정보를 섞는 역할
    return out.astype(np.float32)  # residual connection에서 사용될 과거 토큰들에 대한 attention 가중합 결과 반환

# 한 토큰을 1스텝 앞으로 추론시키는 핵심 루프
def ffn_step(cfg: GPTConfig, blk, x):  # FFN(MLP) 한 번
    # x: (D,)
    h = x @ blk["W1"] + blk["b1"]    # (d_ff,)
    h = gelu(h)
    y = h @ blk["W2"] + blk["b2"]    # (D,)
    return y.astype(np.float32)  # dtype 고정

# 토큰 token_id 하나를 입력으로 받아서, 모든 레이어를 통과시켜 마지막 hidden (D,) 를 만들고, 그 과정에서 KV cache에 현재 토큰의 K/V를 레이어별로 저장한다.
def gpt_step(cfg: GPTConfig, W, token_id: int, pos: int, cache: KVCache):  # Transformer block들을 한 토큰에 대해 한 번 통과 + KV cache 업데이트
    """
    One token step forward; updates cache.
    returns last hidden (D,)
    """
    assert 0 <= token_id < cfg.vocab_size  # token_id 범위 체크
    assert 0 <= pos < cfg.max_seq  # pos 범위 체크
    # 토큰 임베딩
    x = W["tok_emb"][token_id].astype(np.float32)  # W["tok_emb"]는 (V, D), token_id는 정수 하나, W["tok_emb"][token_id]는 해당 토큰의 벡터 (D,)

    for layer in range(cfg.n_layers):   # 같은 구조의 블록이 n_layers번 반복
        blk = W["blocks"][layer]  # blk: 해당 레이어의 파라미터 묶음(dict), W: dict, W["block"]: 레이어별 파라미터 dict들의 리스트, W["blocks"][layer]: W["block"]리스트에서 layer번째 원소
                                  # 리스트안에 dict가 들어있는 구조
        # LN1 -> Attn -> Residual (pre-LayerNorm)
        x_ln = layernorm(x, blk["ln1_g"], blk["ln1_b"], eps=cfg.ln_eps).astype(np.float32)  # LN으로 스케일 안정화
        a = attention_step(cfg, blk, x_ln, pos, cache, layer)  # attiention 계산, a: 이 레이어에서 attention이 만든 출력 벡터
        x = (x + a).astype(np.float32)  # residual 연결

        # LN2 -> FFN -> Residual (pre-LayerNorm)
        x_ln2 = layernorm(x, blk["ln2_g"], blk["ln2_b"], eps=cfg.ln_eps).astype(np.float32)
        f = ffn_step(cfg, blk, x_ln2)  # FFN은 토큰 단위로 비선형 변환을 줘서 표현력을 키우는 역할
        x = (x + f).astype(np.float32) # residual 연결

    x = layernorm(x, W["ln_f_g"], W["ln_f_b"], eps=cfg.ln_eps).astype(np.float32)  # final layernorm: residual 누적으로 분포가 흔들리는 것을 방지하기 위해 최종 hidden을 일관된 분포로 정돈하여 다음 단계 출력 head가 안정적으로 logits을 만듦

    # finished this timestep across all layers
    cache.step_done()  # 이 토큰이 전 레이어를 다 통과했으며, T+=1해줌
    return x  # 이 토큰의 최종 hidden state(D, ), 다음 logits을 만들 때 쓰임

def logits_from_hidden(W, h):  # 마지막 hidden -> vocab logits, h -> vl 변환이 필요한 곳이면 어디든 다시 쓸 수 있음
    # h: (D,), head_W: (D,V)
    return (h @ W["head_W"] + W["head_b"]).astype(np.float32)  # 마지막 hidden h(D, )를 vocab (V, ) 각 토큰에 대한 점수로 변환, 이 logits에 softmax를 씌우면 확률 분포가 되고, top-k / sampling으로 다음 토큰을 뽑게 됨

# -----------------------------
# 7) Generation
# -----------------------------
# prompt를 받아서 토큰을 하나씩 추가 생성하는 루프
def generate(cfg: GPTConfig, W, prompt: str, max_new_tokens=64, temperature=1.0, top_k=0, do_sample=False, stop_on_eos=False):  #prompt: 시작 텍스트, temperature: 확률 분포 "날카로움" 조절,
                                                          #do_sample: True면 확률적으로 랜덤 샘플링, False면 argmax(가장 높은 확률 토큰 선택, 결정적), stop_on_eos: EOS 토큰 나오면 생성 중단할지 여부
    tokens = encode(prompt)  # 토큰화: prompt 문자열 -> 토큰 ID 리스트
    assert len(tokens) <= cfg.max_seq  # prompt 길이가 max_seq 넘지 않게
    cache = KVCache(cfg)  # KV cache 준비

    # prefill: 프롬프트를 캐시에 미리 채우기 -> 프롬프트 토큰들을 한 번씩 gpt_step()에 넣어 KV cache를 쌓는 과정
    h = None
    for pos, tid in enumerate(tokens):  # (인덱스, 값), pos: 토큰의 절대 위치, tid: 그 위치에 있는 토큰 값
        assert 0 <= pos < cfg.max_seq  # pos 범위 체크
        h = gpt_step(cfg, W, tid, pos, cache)  # 마지막 토큰까지 처리한 최종 hidden(D, )

    # generate: 새 토큰을 반복 생성
    for _ in range(max_new_tokens):  # 새 토큰을 최대 토큰 개수만큼 만들겠다
        logits = logits_from_hidden(W, h)

        # temperature
        t = max(1e-6, float(temperature))  # t가 작으면 분포가 더 뾰족 -> 높은 확률 토큰을 더 선택(보수적), t가 크면 분포가 더 평평 -> 다양한 토큰이 나올 가능성 증가(랜덤)
                                           # max(1e-6, ): 0으로 나누기 방지
        logits = logits / t

        # top-k: 확률이 큰 상위 k개만 남기고 나머지 토큰은 logits을 매우 작은 값으로 만듦
        logits = top_k_filter(logits, top_k)

        probs = softmax(logits)

        if do_sample:
            next_id = sample_from_probs(probs)  # 확률분포에서 랜덤 샘플링 생성, 결과가 매번 달라짐
        else:
            next_id = int(np.argmax(probs))  # 가장 확률 높은 토큰 선택(greedy), 항상 같은 결과(결정적)

        assert 0 <= next_id < cfg.vocab_size  # 생성 토큰 범위 체크

        tokens.append(next_id) # 새 토큰을 prompt 뒤에 붙임
        if stop_on_eos and next_id == EOS:  # stop 옵션이 켜져 있고, EOS가 나오면 종료
            break

        pos = len(tokens) - 1 # len(tokens) - 1: 마지막 원소 인덱스이므로 새 토큰의 위치 pos를 계산
        assert pos < cfg.max_seq  # 생성하면서 pos가 max_seq를 넘으면 중단
        h = gpt_step(cfg, W, next_id, pos, cache)  # 새 토큰의 hidden state(h)를 얻고 cache에 K/V도 한 step 추가

    return decode_ascii(tokens), tokens  # 토큰 리스트를 다시 텍스트로 디코딩해서 반환

# -----------------------------
# 8) Demo (Colab)
# -----------------------------
if __name__ == "__main__":  # 실행용 데모
    cfg = GPTConfig(
        n_layers=2,
        d_model=128,  # 토큰 임베딩/히든 차원 개수
        n_heads=4,
        d_ff=512,
        max_seq=256
    )

    # Random weights for structure demo (inference pipeline validation)
    W = init_weights(cfg, seed=123)

    prompt = "Hello! This is a NumPy Transformer. "  # 이 문장 다음에 뭐가 올지 모델이 예측하게 할 시작 텍스트
    out, tokens = generate(  # 텍스트 생성 호출
        cfg, W, prompt,
        max_new_tokens=120,
        temperature=1.0,  # 기본 분포
        top_k=50,
        do_sample=True,  # 확률적 샘플링
        stop_on_eos=False  # EOS 나와도 계속 생성
    )

    print("=== PROMPT ===")
    print(prompt)
    print("\n=== OUTPUT ===")
    print(out)
    print(tokens)


=== PROMPT ===
Hello! This is a NumPy Transformer. 

=== OUTPUT ===
Hello! This is a NumPy Transformer. pFU/ta?u*ObCW1\p[tzGPGvM[?#ppb)j:Pvlow&p
[256, 72, 101, 108, 108, 111, 33, 32, 84, 104, 105, 115, 32, 105, 115, 32, 97, 32, 78, 117, 109, 80, 121, 32, 84, 114, 97, 110, 115, 102, 111, 114, 109, 101, 114, 46, 32, 177, 174, 244, 134, 175, 9, 243, 112, 192, 5, 70, 177, 85, 47, 116, 221, 97, 1, 5, 248, 186, 63, 253, 199, 211, 117, 248, 169, 23, 42, 184, 175, 79, 233, 98, 67, 87, 49, 8, 172, 156, 92, 141, 128, 112, 228, 140, 13, 243, 91, 116, 122, 71, 242, 80, 167, 253, 71, 191, 180, 118, 77, 194, 13, 173, 134, 184, 212, 158, 134, 91, 14, 63, 35, 163, 179, 18, 112, 112, 186, 14, 191, 130, 228, 98, 41, 106, 58, 255, 208, 5, 227, 141, 13, 251, 80, 191, 118, 229, 234, 151, 22, 193, 108, 130, 111, 223, 221, 27, 237, 119, 38, 197, 172, 174, 255, 202, 199, 184, 112]
